# Imports

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from tokenizers import Tokenizer


# Settings

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "Models").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "Models"
LEGACY_TOKENIZER_PATH = MODELS_DIR / "Tokenizer" / "tokenizer_30000.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Model Definition

In [ ]:
class SimpleTransformer(nn.Module):

    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, seq_len):

        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(seq_len, embed_dim)

        self.transformer_blocks = nn.ModuleList([nn.TransformerEncoderLayer(d_model=embed_dim,
                                                                            nhead=num_heads,
                                                                            
                                                                            dim_feedforward=hidden_dim,
                                                                            batch_first=False) for _ in range(num_layers)])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)

        x = self.embedding(x) + self.position_embedding(positions)
        x = x.transpose(0, 1) 
        
        # Ajout du masque causal
        mask = torch.triu(torch.ones(x.size(0), x.size(0), device=x.device) * float('-inf'), diagonal=1)
        
        for block in self.transformer_blocks:
            x = block(x, src_mask=mask)
        
        x = x.transpose(0, 1)  
        x = self.norm(x)
        
        return self.fc(x)

# Tokenizer Function

In [ ]:
def load_tokenizer(tokenizer_path):

    tokenizer_path = Path(tokenizer_path)

    if not tokenizer_path.is_file():
        raise FileNotFoundError(f"Tokenizer not found: {tokenizer_path}")
    
    return Tokenizer.from_file(str(tokenizer_path))

# Generate Function

In [ ]:
def generate_text(model, tokenizer, prompt, seq_len, max_length=100, temperature=0.9, top_k=50, top_p=0.92, device=DEVICE, stop_at_eos=True):

    model.eval()

    eos_token_id = tokenizer.token_to_id("<eos>")
    pad_token_id = tokenizer.token_to_id("<pad>")
    user_token_id = tokenizer.token_to_id("<user>")
    assistant_token_id = tokenizer.token_to_id("<assistant>")
    
    input_ids = tokenizer.encode(prompt).ids
    input_ids = input_ids[-seq_len:] if len(input_ids) > seq_len else input_ids
    
    input_tensor = torch.tensor([input_ids], device=device)
    generated_tokens = []
    
    with torch.no_grad():

        for step in range(max_length):

            logits = model(input_tensor)
            next_token_logits = logits[0, -1, :] / temperature
            
            probs = F.softmax(next_token_logits, dim=-1)
    
            if top_k > 0:

                top_k_probs, top_k_indices = torch.topk(probs, min(top_k, probs.size(-1)))
                probs = torch.zeros_like(probs)

                probs.scatter_(0, top_k_indices, top_k_probs)
                probs = probs / probs.sum()
            
            if top_p < 1.0:

                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                
                sorted_indices_to_remove = cumulative_probs > top_p

                sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
                sorted_indices_to_remove[0] = False
                
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                probs[indices_to_remove] = 0
                probs = probs / probs.sum()

            next_token = torch.multinomial(probs, num_samples=1).unsqueeze(0)
            
            if next_token.item() in [eos_token_id, pad_token_id]:
                break
            
            generated_tokens.append(next_token)
            input_tensor = torch.cat([input_tensor, next_token], dim=1)
            
            if input_tensor.size(1) > seq_len:
                input_tensor = input_tensor[:, -seq_len:]

    generated_text = tokenizer.decode(generated_tokens)
    return generated_text

# Load Model and Settings

In [ ]:
def select_model():

    model_files = sorted(path for path in MODELS_DIR.rglob("*.pth") if "checkpoint" not in path.name.lower())

    if not model_files:
        raise FileNotFoundError(f"No model files found under {MODELS_DIR}. Train a model first.")

    print("\nAvailable Models:")
    
    for index, model_path in enumerate(model_files, 1):
        print(f"{index:2d}. {model_path.relative_to(MODELS_DIR)}")

    while True:
        
        try:
            choice = int(input("\nChoose a model number: "))
            
            if 1 <= choice <= len(model_files):
                model_path = model_files[choice - 1]
                settings_path = model_path.with_name(f"{model_path.stem}_settings.json")
                
                if not settings_path.is_file():
                    raise FileNotFoundError(f"Missing model settings: {settings_path}")
                
                return model_path, settings_path
            
            print("Invalid choice. Please try again.")
        
        except ValueError:
            print("Please enter a valid number.")
        
        except KeyboardInterrupt:
            print("\nExiting...")
            raise

# Interact with the LLM

In [ ]:
print(f"\n{'=' * 60}\nPROMPTED TEXT GENERATION\n{'=' * 60}")

model_path, settings_path = select_model()

with open(settings_path, "r", encoding="utf-8") as file:
    settings = json.load(file)

tokenizer_path = PROJECT_ROOT / settings.get("TOKENIZER_PATH", str(LEGACY_TOKENIZER_PATH.relative_to(PROJECT_ROOT)))
tokenizer = load_tokenizer(tokenizer_path)

model = SimpleTransformer(vocab_size=settings["VOCAB_SIZE"],
                          embed_dim=settings["EMBED_DIM"],
                          num_heads=settings["NUM_HEADS"],

                          hidden_dim=settings["HIDDEN_DIM"],
                          num_layers=settings["NUM_LAYERS"],
                          seq_len=settings["SEQ_LEN"]).to(DEVICE)

model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
model.eval()

print(f"Loaded {model_path.relative_to(MODELS_DIR)} on {DEVICE}")

print("\nType a message to generate a continuation.")
print("Commands: 'quit'/'exit' to leave, 'clear' to reset context, 'params' to adjust settings.")

gen_params = {"top_k": 40, "top_p": 0.90, "max_length": 80, "temperature": 0.8}
conversation_history = ""

def format_prompt(history, user_message):
    return f"{history}<user> {user_message} <eos>\n<assistant>"

def append_conversation_turn(history, user_message, assistant_message, tokenizer, context_length):
    
    turn = f"<user> {user_message} <eos>\n<assistant> {assistant_message} <eos>\n"
    token_ids = tokenizer.encode(history + turn).ids[-context_length:]
    
    return tokenizer.decode(token_ids)

while True:

    try:
        user_input = input("You: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() in ("quit", "exit", "escape"):
            print("\nGoodbye!")
            break
        
        if user_input.lower() == "clear":
            conversation_history = ""
            print("Context cleared.\n")
            continue
        
        if user_input.lower() == "params":
            print(f"Current parameters: {gen_params}")
            
            try:
                updated_params = gen_params.copy()
                
                for key, prompt, convert in (("top_k", "New top-k: ", int),
                                             ("top_p", "New top-p: ", float),
                                             ("max_length", "New max length: ", int),
                                             ("temperature", "New temperature: ", float)):
                    
                    value = input(prompt).strip()
                    
                    if value:
                        updated_params[key] = convert(value)
                
                if updated_params["top_k"] < 1 or not 0 < updated_params["top_p"] <= 1:
                    raise ValueError("top_k must be positive and top_p must be in (0, 1]")
                
                if updated_params["max_length"] < 1 or updated_params["temperature"] <= 0:
                    raise ValueError("max_length and temperature must be positive")
                
                gen_params = updated_params
            
            except ValueError as error:
                print(f"Invalid parameters: {error}")
            
            continue

        prompt = format_prompt(conversation_history, user_input)
        
        response = generate_text(model=model,
                                 tokenizer=tokenizer,

                                 prompt=prompt,
                                 seq_len=settings["SEQ_LEN"],
                                 max_length=gen_params["max_length"],
                                 temperature=gen_params["temperature"],

                                 top_k=gen_params["top_k"],
                                 top_p=gen_params["top_p"],
                                 device=DEVICE).strip()
        
        response = response.replace("<user>", "").replace("<assistant>", "").replace("<eos>", "").strip()
        print(f"Assistant: {response}\n")

        conversation_history = append_conversation_turn(conversation_history, user_input, response, tokenizer, settings["SEQ_LEN"])

    except KeyboardInterrupt:
        print("\nGoodbye!")
        break
    
    except Exception as error:
        print(f"\nGeneration error: {error}\n")